<a href="https://colab.research.google.com/github/nurgu1/MachineLearning1_sem/blob/main/class5_GridSearchCV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GridSearch in Machine Learning

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
uciml_breast_cancer_wisconsin_data_path = kagglehub.dataset_download('uciml/breast-cancer-wisconsin-data')

print('Data source import complete.')


Using Colab cache for faster access to the 'breast-cancer-wisconsin-data' dataset.
Data source import complete.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

In [ ]:
data = pd.read_csv('/kaggle/input/breast-cancer-wisconsin-data/data.csv')

In [ ]:
data.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [ ]:
X = data.drop(['id', 'diagnosis', 'Unnamed: 32'], axis = 1)

y = data['diagnosis']

In [ ]:
# X.isnull().sum()

In [ ]:
X.shape, y.shape

((569, 30), (569,))

# Build Logistic Regression
Lets first Build Logistic Regression Model.

In [ ]:
# lr = LogisticRegression()
lr = LogisticRegression(max_iter=5000)

In [ ]:
lr.fit(X,y)

LogisticRegression(max_iter=5000)

# Check Accuracy

In [ ]:
lr.score(X,y)

0.9578207381370826

# Build Logistic Regression with Hyperparameter
Now lets build the Logistic Regression model with Hyperparameter, and will be using GridSearchCV to achive this.

Defining the hyper-parameters.

The details on these parameters can be checked from https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

In [ ]:
params = [
    {'penalty' : ['l1', 'l2', 'elasticnet', 'none'],   # Used to specify the norm used in the penalization.
    'C' : np.logspace(-4, 4, 20),                      # Inverse of regularization strength; must be a positive float.
    'solver' : ['lbfgs','newton-cg','liblinear','sag','saga'],  # Algorithm to use in the optimization problem.
    'max_iter' : [100, 1000,2500, 5000]                # Maximum number of iterations taken for the solvers to converge.
    }
]

# There are many other parameters that we could use... but for nw will start with this.

As we will be using GridSearchCV we have to import it first.

For more details on GridSearchCV, refer https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html?highlight=gridsearchcv#sklearn.model_selection.GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
clf = GridSearchCV(estimator = lr, param_grid = params, scoring = 'accuracy', cv = 3, verbose=True, n_jobs=-1)
# cv --> Determines the cross-validation splitting strategy
# verbose --> Controls the verbosity. Verbose is a general programming term for produce lots of logging output. You can think of it as asking the program to "tell me everything about what you are doing all the time".
# n_jobs --> Number of jobs to run in parallel. `-1` means using all processors.

In [ ]:
clf_fit = clf.fit(X,y)

Fitting 3 folds for each of 1600 candidates, totalling 4800 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
3120 fits failed out of a total of 4800.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
240 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py", line 1193, in fit
    solver = _check_s

In [ ]:
# Estimator that was chosen by the search, i.e. estimator which gave highest score (or smallest loss if specified) on the left out data.
clf_fit.best_estimator_

LogisticRegression(C=np.float64(1438.44988828766), max_iter=5000)

In [ ]:
clf_fit.score(X,y)
# Returns the score on the given data.
# This uses the score defined by scoring where provided, and the best_estimator_.score method otherwise.

0.9876977152899824

In [ ]:
# Mean cross-validated score of the best_estimator
clf_fit.best_score_

np.float64(0.9648565859092174)

So we have seen that the Logistic Regression has resulted as ~95% but with the Hyper-Parameter it has scored as ~98%.

In [ ]:
clf_fit.best_params_

{'C': np.float64(1438.44988828766),
 'max_iter': 5000,
 'penalty': 'l2',
 'solver': 'lbfgs'}

In [ ]:
clf_fit.best_estimator_

LogisticRegression(C=np.float64(1438.44988828766), max_iter=5000)